Compare precomputed and on-the-go mz grids


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mascope_lib.file_func import load_file
import json

In [ ]:
f = load_file(
    "TestOrbitrap_20231227_1010_MION2_DBrMe_DHS_1ul_50pg_Explosive_Mix_60-600mz"
)
f_mz_prebuilt = load_file(
    "NewTestOrbitrap_20231227_1010_MION2_DBrMe_DHS_1ul_50pg_Explosive_Mix_60-600mz"
)

In [ ]:
data = {
    "old": {
        "2D": f.signal.dropna(dim="mz", how="all").values,
        "sum": f.signal.dropna(dim="mz", how="all").sum(dim="time").values,
        "mz": f.signal.dropna(dim="mz", how="all").mz.values,
    },
    "new": {
        "2D": f_mz_prebuilt.signal.dropna(dim="mz", how="all").values,
        "sum": f_mz_prebuilt.signal.dropna(dim="mz", how="all").sum(dim="time").values,
        "mz": f_mz_prebuilt.signal.dropna(dim="mz", how="all").mz.values,
    },
}

In [ ]:
key = "old"
mz_down, mz_up = 72.9928, 72.9935
# mz_down, mz_up = 363.078, 363.085

fig, axes = plt.subplots(ncols=2, figsize=(12, 3))


mz_mask = np.where((data[key]["mz"] >= mz_down) & (data[key]["mz"] <= mz_up))
mz_scale = data[key]["mz"][mz_mask]

spec_slices = data[key]["2D"][mz_mask]

# Plot each 10th spectrum from time dim
for i in np.arange(0, 50, 10):
    try:
        axes[0].scatter(x=mz_scale, y=spec_slices[:, i])
    except ValueError:
        print(i)
# Plot vertical mz lines
for i in mz_scale:
    axes[0].axvline(x=i, alpha=0.5)
# Plot sum spectrum
axes[1].plot(mz_scale, data[key]["sum"][mz_mask], marker="o")

# Style
axes[0].set(title="Raw datapoints", xlabel="m/z [Th]", ylabel="Counts [a.u.]")
axes[1].set(title="Sum", xlabel="m/z [Th]", ylabel="Counts [a.u.]")

plt.savefig(
    f"C:\\Users\\alvai\\Desktop\\pics\\{key}, mz={mz_down, mz_up}.png",
    dpi=300,
    bbox_inches="tight",
)

Test precomputation of mz for detect_peaks()


In [ ]:
f = load_file(
    "TestOrbitrap_20231227_1010_MION2_DBrMe_DHS_1ul_50pg_Explosive_Mix_60-600mz"
)

In [ ]:
old_mz = f.signal.mz.values

# Initialize an empty array to store the closest values
mz_closest = np.empty_like(old_mz)

# Iterate over the elements in mz and find the closest value from mz_grid
temp_mz = 0


for i, val in enumerate(old_mz):
    if temp_mz != np.round(val):
        temp_mz = np.round(val)
        grid_slice = mz_grid[
            np.where((mz_grid > temp_mz - 0.5) & (mz_grid < temp_mz + 0.5))
        ]
    closest_val = grid_slice[np.abs(grid_slice - val).argmin()]
    mz_closest[i] = closest_val

In [ ]:
def precompute_grid(mz_min, mz_max, points_per_peak):
    """Precompute mz grid"""

    # TODO proper way to get resolution function

    def R_orb(mz):
        return 1765522.209284122 / np.sqrt(mz)

    def get_fwhm(mz):
        return mz / R_orb(mz)

    mz_grid = []
    mz = mz_min
    while mz < mz_max:
        fwhm = get_fwhm(mz)
        step = fwhm / points_per_peak
        mz_grid.append(mz + step)

        mz += step
    return np.array(mz_grid)

In [ ]:
mz_grid = precompute_grid(50, 650, 3)

In [ ]:
def _set_mz_precision(mz, spec):
    # Find the indices of the closest values from self._mz_grid
    closest_indices = np.searchsorted(mz_grid, mz)
    # Get the closest mz values from self._mz_grid
    mz_closest = mz_grid[closest_indices]
    # Get unique mz values and their corresponding indices
    unique_mz, unique_mz_indices = np.unique(mz_closest, return_inverse=True)
    # Accumulate the intensities for each unique mz value
    unique_mz_intensities = np.bincount(closest_indices, weights=spec)

    return unique_mz, unique_mz_intensities

In [ ]:
mz_grid = precompute_grid(50, 650, 3)

mz = np.sort(np.random.uniform(50, 600, size=10000))
spec = np.random.uniform(0, 1e6, size=10000)


# Find the indices of the closest values from self._mz_grid
closest_indices = np.searchsorted(mz_grid, mz)
# Get the closest mz values from self._mz_grid
mz_closest = mz_grid[closest_indices]
# Get unique mz values and their corresponding indices
unique_mz, inverse_indices = np.unique(mz_closest, return_inverse=True)
# Accumulate the intensities for each unique mz value
unique_mz_intensities = np.bincount(inverse_indices, weights=spec)

In [ ]:
plt.plot(unique_mz, unique_mz_intensities)

Test peak shapes at different intensities


In [ ]:
# Density of intensities
# range: (all peaks, "good" peaks)
intensities = {
    "1e7, 1e8": (14, 12),
    "1e6, 1e7": (69, 24),
    "1e5, 1e6": (345, 39),
    "1e4, 1e5": (482, 21),
    "1e3, 1e4": (626, 14),
    "1e2, 1e3": (6384, 17),
}

# Sort the keys in ascending order
sorted_keys = sorted(intensities.keys())

# Extract the values for the first and second elements in each tuple
first_values = [intensities[key][0] for key in sorted_keys]
second_values = [intensities[key][1] for key in sorted_keys]

# Plot the barplot with a logarithmic scale for the y-axis
plt.bar(sorted_keys, first_values, color="blue", alpha=0.5, label="Total peaks")
plt.bar(
    sorted_keys,
    second_values,
    color="orange",
    alpha=0.7,
    label="Fraction used for a median peak shape",
)

# Set a logarithmic scale for the y-axis
plt.yscale("log")

# Add labels and title
plt.xlabel("mz range")
plt.ylabel("Number of peaks (log scale)")
plt.title("Distribution of peaks per intensity range")

# Add legend
plt.legend()

# Show the plot
# plt.savefig(f"C:\\Users\\alvai\\Desktop\\Distribution of peaks per intensity range.png",
#             dpi=200,
#             bbox_inches="tight")

In [ ]:
peak_shapes = {
    "1e7, 1e8": "1.0e+07...1.0e+08.json",
    "1e6, 1e7": "1.0e+06...1.0e+07.json",
    "1e5, 1e6": "1.0e+05...1.0e+06.json",
    "1e4, 1e5": "1.0e+04...1.0e+05.json",
    "1e3, 1e4": "1.0e+03...1.0e+04.json",
    "1e2, 1e3": "1.0e+02...1.0e+03.json",
}
for key, value in peak_shapes.items():
    with open(f"C:\\Users\\alvai\\Desktop\\peak_shapes\\{value}") as f:
        peak_shapes[key] = json.load(f)

In [ ]:
# Plot y vs x for each key in peak_shapes
plt.figure(figsize=(8, 3))

for key, values in peak_shapes.items():
    x = values["x"]
    y = values["y"]
    plt.plot(x, y, label=key, linewidth=2.5)

# Configure plot
plt.xlim(-3, 3)
plt.xlabel("Normalized mz range")
plt.ylabel("Normalized counts")
plt.title("Median peak shape at different peak intensity ranges")

plt.legend(title="Intensity range")
plt.savefig(
    f"C:\\Users\\alvai\\Desktop\\Peak shape change with intensity range.png",
    dpi=200,
    bbox_inches="tight",
)

Test resolution function change from file to file


In [ ]:
def R_orb(a, mz):
    return a / np.sqrt(mz)


artif_mz_range = np.arange(200, 400)

res_fun_coef = {
    "40-200mz": 1709329.725,
    "150-2000mz": 1718014.501,
    "60-600mz": 1718583.189,
    "50-500mz": 1714021.872,
    "60-600mz": 1718800.813,
    "Approximation": 1.715e6,
}

plt.figure(figsize=(7, 3))

for key, val in res_fun_coef.items():
    if key == "Approximation":
        plt.plot(
            artif_mz_range,
            R_orb(val, artif_mz_range),
            label=key,
            linewidth=2,
            linestyle="--",
            color="black",
        )
    else:
        plt.plot(artif_mz_range, R_orb(val, artif_mz_range), label=key)

plt.legend()
plt.xlabel("mz")
plt.ylabel("Resolution function")